# Phase 6: Creative Challenge
### Improving learning on FER2013 dataset using different techniques

### Setup

In [5]:
import os, sys, time, hashlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
FER_DIR = ROOT / "data" / "fer"
FEAT_DIR  = os.path.join(FER_DIR, "features")
FIG_DIR = ROOT / "figures"
MODEL_DIR = ROOT / "models"

from scripts.fer_features import EMOTIONS, FEATURE_FNS

warnings.filterwarnings("ignore", category=FutureWarning)

# ----- settings --------------------------------------------------------------------------
RANDOM_STATE = 42
ABL_CLF      = "LogReg"       # fast classifier used for the ablation steps (seconds per run)
MIN_GAIN     = 0.002          # a step is kept if validation macro-F1 improves by at least this
FINAL_CLF    = "SVM"          # classifier for the final model...
FINAL_PARAMS = {'C':5,'gamma':0.001}             # ...with the tuned values from Task 1, e.g. {"C": 5, "gamma": 0.0005}
RUN_FINAL_TEST = False        # True ONCE, at the very end

### Helpers

In [6]:
def load_split(name):
    df = pd.read_csv(os.path.join(FER_DIR, f"fer2013_{name}.csv"))
    return np.stack([np.array(p.split(), dtype=np.uint8).reshape(48, 48) for p in df["pixels"]]), df["emotion"].values

def cached_features(feature, images, split):
    cache_name, extract = FEATURE_FNS[feature]
    path = os.path.join(FEAT_DIR, f"{cache_name}_{split}.npy")
    if os.path.exists(path): return np.load(path)
    X = extract(images); np.save(path, X); return X

def scores(y_true, y_pred):
    return dict(accuracy=accuracy_score(y_true, y_pred),
                balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
                macro_f1=f1_score(y_true, y_pred, average="macro"))

def make_model(clf, class_weight=None, **params):
    """Scaler -> classifier. `params` override defaults (use the tuned values from Task 1 for the final model)."""
    steps = [("scaler", StandardScaler())] if clf in ("LogReg", "SVM") else []
    if clf == "LogReg":
        kw = dict(C=1.0, max_iter=1000, class_weight=class_weight, random_state=RANDOM_STATE); kw.update(params)
        steps.append(("clf", LogisticRegression(**kw)))
    elif clf == "SVM":
        kw = dict(kernel="rbf", C=1.0, gamma="scale", cache_size=1000, class_weight=class_weight, random_state=RANDOM_STATE); kw.update(params)
        steps.append(("clf", SVC(**kw)))
    else:
        kw = dict(n_estimators=300, max_features="sqrt", n_jobs=-1, class_weight=class_weight, random_state=RANDOM_STATE); kw.update(params)
        steps.append(("clf", RandomForestClassifier(**kw)))
    return Pipeline(steps)

img_train, y_train = load_split("train")
img_val,   y_val   = load_split("validation")
print(f"train {img_train.shape}, validation {img_val.shape}")

train (28709, 48, 48), validation (3589, 48, 48)
